# Learning Rate Scheduling and Advanced Techniques

**Author:** Siddharth Bokka  
**Dataset:** CIFAR-10 (CNN training)  
**Frameworks:** Pure Python + PyTorch + MLflow  

---

## What This Notebook Covers

| Technique | Section |
|---|---|
| LR Schedule Visualization | Step 2 |
| StepLR vs Cosine vs OneCycle vs Warmup | Step 3 |
| Gradient Clipping — why and how | Step 4 |
| Batch Size Effect on Convergence | Step 5 |
| Weight Initialization — Xavier vs He | Step 6 |
| Scheduler Comparison on CIFAR-10 (MLflow) | Step 7 |

---

**The core insight of this notebook:**  
The optimizer (Adam, AdamW) is only half the story.  
*How* the learning rate changes during training determines  
whether the model converges to a sharp or flat minimum,  
how fast it gets there, and whether it overfits.

## Step 1 — Setup

In [ ]:
import sys
import logging
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import torch

repo_root = Path().resolve().parent
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

logging.basicConfig(
    format='%(asctime)s | %(levelname)-8s | %(name)s | %(message)s',
    datefmt='%H:%M:%S',
    level=logging.INFO,
)

matplotlib.rcParams.update({'figure.dpi': 120, 'font.size': 12})
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')

In [ ]:
from gdo.optimizers.schedulers import (
    StepLR, CosineAnnealingLR, OneCycleLR, CyclicalLR,
    WarmupScheduler, ReduceLROnPlateau
)
from gdo.landscapes import LandscapePlotter
from gdo.config import ExperimentConfig
from gdo.training import Trainer
from gdo.experiment import ExperimentLogger

print('gdo imports successful.')

---
## Step 2 — Visualize All Schedules Side-by-Side

The most intuitive way to understand a scheduler is to plot the LR curve  
it produces over 30 training epochs.

In [ ]:
BASE_LR = 0.001
TOTAL_EPOCHS = 30

schedules = {
    'Constant (no scheduler)': np.full(TOTAL_EPOCHS, BASE_LR),
    'StepLR (÷10 every 10 epochs)': StepLR(BASE_LR, step_size=10, gamma=0.1).get_lr_curve(TOTAL_EPOCHS),
    'CosineAnnealingLR': CosineAnnealingLR(BASE_LR, t_max=TOTAL_EPOCHS).get_lr_curve(TOTAL_EPOCHS),
    'OneCycleLR (max=0.01)': OneCycleLR(BASE_LR, max_lr=0.01, total_epochs=TOTAL_EPOCHS, pct_start=0.3).get_lr_curve(TOTAL_EPOCHS),
    'CyclicalLR (triangular2)': CyclicalLR(BASE_LR, max_lr=0.01, step_size_up=4, mode='triangular2').get_lr_curve(TOTAL_EPOCHS),
    'Warmup + Cosine': WarmupScheduler(BASE_LR, total_epochs=TOTAL_EPOCHS, warmup_steps=5).get_lr_curve(TOTAL_EPOCHS),
}

fig = LandscapePlotter.lr_schedule_plot(
    schedules=schedules,
    title='Learning Rate Schedules — 30 Epoch Comparison',
    figsize=(12, 5),
)
plt.show()

### What to observe

- **StepLR**: LR drops suddenly at fixed intervals — creates training instability  
  around step boundaries, especially with large gamma.

- **CosineAnnealingLR**: Smooth monotonic decay. The cosine shape means LR  
  decreases slowly at first, then quickly in the middle, then slowly again —  
  matching the typical shape of loss improvement.

- **OneCycleLR**: Starts low, warms up to a high LR, then decays.  
  The high LR phase acts as a form of regularization, helping escape  
  sharp minima. This is the fast.ai training recipe.

- **CyclicalLR**: Oscillates between min and max LR.  
  The periodic increases prevent getting stuck in sharp minima.  
  `triangular2` halves the amplitude each cycle.

- **Warmup + Cosine**: Standard for Transformers. Cold-start instability  
  is avoided by linearly increasing LR in the first few epochs.

---
## Step 3 — Why Warmup Matters: The Cold-Start Problem

In [ ]:
# Simulate gradient norms at the start of training — before and after warmup
# In real training, gradients are large early on (loss surface is steep near random init)
# Starting with a high LR multiplies these large gradients into huge updates.

np.random.seed(SEED)
n_steps = 50

# Simulated gradient norms at the start of training (decaying over time)
grad_norms = 5.0 * np.exp(-np.linspace(0, 3, n_steps)) + np.random.normal(0, 0.2, n_steps)

# No warmup: constant LR
lr_no_warmup = np.full(n_steps, BASE_LR)

# With warmup: linear ramp over first 10 steps
warmup_steps = 10
lr_warmup = np.concatenate([
    np.linspace(BASE_LR * 0.01, BASE_LR, warmup_steps),
    np.full(n_steps - warmup_steps, BASE_LR)
])

# Effective gradient update magnitude = lr * grad_norm
update_no_warmup = lr_no_warmup * grad_norms
update_warmup = lr_warmup * grad_norms

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(grad_norms, label='Gradient Norm', color='gray', linewidth=2)
ax1.set_title('Simulated Gradient Norms\n(large at init, decays)', fontsize=12)
ax1.set_xlabel('Step'); ax1.set_ylabel('Grad Norm'); ax1.grid(alpha=0.3)

ax2.plot(update_no_warmup, label='No warmup', color='#d62728', linewidth=2)
ax2.plot(update_warmup, label='With warmup', color='#2ca02c', linewidth=2)
ax2.set_title('Effective Update Magnitude\n(lr × grad_norm)', fontsize=12)
ax2.set_xlabel('Step'); ax2.set_ylabel('Update Magnitude'); ax2.legend(); ax2.grid(alpha=0.3)

fig.suptitle('Cold-Start Problem: Without Warmup, Early Updates Are 100× Too Large', fontsize=13)
fig.tight_layout()
plt.show()

print(f'Max update without warmup: {update_no_warmup.max():.4f}')
print(f'Max update with warmup:    {update_warmup.max():.4f}')
print(f'Ratio: {update_no_warmup.max() / update_warmup.max():.1f}×')

---
## Step 4 — Gradient Clipping: Preventing Exploding Gradients

In RNNs and deep networks, gradients can grow exponentially through many layers  
— the **exploding gradient** problem. Gradient clipping caps the global gradient  
norm to a maximum value before the optimizer step.

$$\text{if } \|g\| > \text{max\_norm}: \quad g \leftarrow g \cdot \frac{\text{max\_norm}}{\|g\|}$$

In [ ]:
# Demonstrate gradient clipping numerically

def clip_grad_norm(grads: np.ndarray, max_norm: float) -> tuple[np.ndarray, float, float]:
    """Clip gradient vector to max_norm. Returns clipped grads, original norm, clipped norm."""
    norm = float(np.linalg.norm(grads))
    if norm > max_norm:
        scale = max_norm / norm
        return grads * scale, norm, max_norm
    return grads, norm, norm

np.random.seed(SEED)

max_norm = 1.0
cases = [
    ('Normal gradient',   np.array([0.3, -0.2, 0.1])),
    ('Large gradient',    np.array([5.0, -3.0, 4.0])),
    ('Exploding gradient',np.array([50.0, -80.0, 30.0])),
]

print(f'{'Case':<22} {'Original Norm':<16} {'After Clipping':<16} {'Was Clipped?'}')
print('-' * 65)
for name, grads in cases:
    clipped, orig_norm, clipped_norm = clip_grad_norm(grads, max_norm)
    was_clipped = orig_norm > max_norm
    print(f'{name:<22} {orig_norm:<16.4f} {clipped_norm:<16.4f} {"YES" if was_clipped else "no"}')

---
## Step 5 — Batch Size Effect on Gradient Noise

Larger batches produce lower-variance gradient estimates — the law of large numbers.  
This changes the **effective learning rate** and the types of minima found.

**The linear scaling rule** (Goyal et al., 2017):  
If batch size is multiplied by $k$, scale the learning rate by $k$  
to maintain the same effective update magnitude.

**Sharp vs flat minima** (Keskar et al., 2017):  
Large-batch training tends to find sharper minima that generalize worse.  
Small-batch training finds flatter minima that generalize better —  
the gradient noise acts as implicit regularization.

In [ ]:
# Simulate gradient variance as a function of batch size
# Var(mean of B samples) = Var(individual sample) / B

batch_sizes = [1, 8, 32, 128, 512, 2048]
per_sample_variance = 1.0  # normalized

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Variance vs batch size
variances = [per_sample_variance / b for b in batch_sizes]
axes[0].loglog(batch_sizes, variances, 'o-', color='#1f77b4', linewidth=2, markersize=8)
axes[0].set_title('Gradient Variance vs Batch Size\n(log-log scale)', fontsize=12)
axes[0].set_xlabel('Batch Size'); axes[0].set_ylabel('Gradient Variance')
axes[0].grid(True, alpha=0.3)
for b, v in zip(batch_sizes, variances):
    axes[0].annotate(f'B={b}', (b, v), textcoords='offset points', xytext=(5, 5), fontsize=9)

# LR scaling rule: lr should scale linearly with batch size
base_bs = 32
base_lr = 0.001
scaled_lrs = [base_lr * (b / base_bs) for b in batch_sizes]
axes[1].semilogx(batch_sizes, scaled_lrs, 's-', color='#d62728', linewidth=2, markersize=8)
axes[1].axvline(base_bs, color='gray', linestyle='--', alpha=0.5, label=f'Base (B={base_bs}, lr={base_lr})')
axes[1].set_title(f'Linear LR Scaling Rule\n(base: B={base_bs}, lr={base_lr})', fontsize=12)
axes[1].set_xlabel('Batch Size'); axes[1].set_ylabel('Recommended LR')
axes[1].legend(fontsize=10); axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

---
## Step 6 — Weight Initialization: Xavier vs He

Bad initialization causes gradients to vanish or explode in the first backward pass —  
before any learning has happened.

**Xavier / Glorot (2010):** $W \sim \mathcal{U}(-\sqrt{6/(n_{in}+n_{out})}, +\sqrt{6/(n_{in}+n_{out})})$  
Designed for tanh/sigmoid activations. Keeps variance constant across layers.

**He / Kaiming (2015):** $W \sim \mathcal{N}(0, \sqrt{2/n_{in}})$  
Designed for ReLU activations. Accounts for ReLU zeroing half the inputs.

In [ ]:
import torch.nn as nn

n_in = 512
n_out = 512

# Simulate forward pass activation variance with different inits
n_layers = 20
batch = torch.randn(64, n_in)

def measure_activation_variance(init_fn: str, activation: str) -> list[float]:
    torch.manual_seed(SEED)
    x = batch.clone()
    variances = [float(x.var())]
    for _ in range(n_layers):
        W = torch.empty(n_in, n_out)
        if init_fn == 'xavier':
            nn.init.xavier_uniform_(W)
        elif init_fn == 'he':
            nn.init.kaiming_normal_(W, nonlinearity='relu')
        elif init_fn == 'naive':
            nn.init.normal_(W, std=0.01)
        x = x @ W
        if activation == 'relu':
            x = torch.relu(x)
        elif activation == 'tanh':
            x = torch.tanh(x)
        variances.append(float(x.var()))
    return variances

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for init_name, color in [('xavier', '#1f77b4'), ('he', '#2ca02c'), ('naive', '#d62728')]:
    var_tanh = measure_activation_variance(init_name, 'tanh')
    var_relu = measure_activation_variance(init_name, 'relu')
    axes[0].semilogy(var_tanh, label=f'{init_name} + tanh', color=color, linewidth=2)
    axes[1].semilogy(var_relu, label=f'{init_name} + ReLU', color=color, linewidth=2)

for ax, title in zip(axes, ['Tanh Activation', 'ReLU Activation']):
    ax.set_title(f'Activation Variance Across {n_layers} Layers\n({title})', fontsize=12)
    ax.set_xlabel('Layer'); ax.set_ylabel('Variance (log scale)')
    ax.axhline(1.0, color='black', linestyle='--', alpha=0.4, label='Ideal (variance=1)')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

print('Observation: naive init causes variance to collapse/explode within a few layers.')
print('Xavier preserves variance with tanh. He preserves variance with ReLU.')

---
## Step 7 — Scheduler Comparison on CIFAR-10 (MLflow)

Run the CNN on CIFAR-10 with different schedulers.  
All logged to the `scheduler-comparison-cifar10` MLflow experiment.

In [ ]:
import yaml

configs_dir = repo_root / 'configs'
base_cfg_path = configs_dir / 'scheduler_comparison.yaml'

schedulers_to_compare = ['none', 'step', 'cosine', 'onecycle', 'warmup_cosine']
scheduler_results = {}

for sched_name in schedulers_to_compare:
    # Load base config and patch scheduler name
    with open(base_cfg_path) as f:
        cfg_dict = yaml.safe_load(f)
    cfg_dict['scheduler']['name'] = sched_name
    cfg_dict['mlflow']['run_name'] = f'adamw_{sched_name}_cifar10'

    cfg = ExperimentConfig(**cfg_dict)

    print(f'\n{'='*55}')
    print(f'Scheduler: {sched_name}')
    print(f'{'='*55}')

    trainer = Trainer.from_config(cfg)
    with ExperimentLogger(cfg.mlflow) as log:
        result = trainer.fit()
        log.log_run(result, cfg)

    scheduler_results[sched_name] = result
    print(f'Done — best_val_acc={result.best_val_acc:.4f} at epoch {result.best_epoch}')

In [ ]:
# Comparison table
print(f'{'Scheduler':<20} {'Best Val Acc':<15} {'Best Epoch':<12} {'Time (s)'}')
print('-' * 60)
for name, result in scheduler_results.items():
    print(f'{name:<20} {result.best_val_acc:<15.4f} {result.best_epoch:<12} {result.total_train_time_s:.1f}')

# Plot val accuracy curves
fig, ax = plt.subplots(figsize=(11, 5))
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
for i, (name, result) in enumerate(scheduler_results.items()):
    ax.plot(result.val_accs, label=name, color=colors[i], linewidth=2)

ax.set_title('CIFAR-10 CNN — Scheduler Comparison (Val Accuracy)', fontsize=13)
ax.set_xlabel('Epoch'); ax.set_ylabel('Val Accuracy')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

---
## Key Takeaways

1. **Constant LR** rarely gives the best result — the right LR at the start  
   is not the right LR at convergence.

2. **Cosine annealing** is a strong default for most tasks — smooth decay  
   without hyperparameter tuning.

3. **OneCycleLR** often reaches the same final accuracy faster than cosine  
   because the high-LR warmup phase acts as exploration.

4. **Warmup + Cosine** is the Transformer training standard. Use it  
   with AdamW for any BERT/GPT-style training.

5. **Gradient clipping** is not optional for RNNs and deep networks —  
   set `grad_clip=1.0` as a default.

6. **Weight initialization** is the first thing that can go wrong.  
   Use He init for ReLU networks. Xavier for tanh. Wrong init can  
   make training impossible regardless of optimizer or scheduler.

7. **Batch size and LR are coupled.** If you double the batch size,  
   double the learning rate (linear scaling rule). This is why distributed  
   training requires careful LR tuning.

---
**All results are logged in MLflow.** Run `mlflow ui` from the repo root  
to compare all experiments in a structured table with metric plots.